In [98]:
!pip install chromadb

In [140]:
import chromadb
chroma_client = chromadb.Client()

In [139]:
collection = chroma_client.get_or_create_collection(name="Documents")

In [101]:
"""collection.add(
    ids=["id1", "id2"],
    documents=[
        "This is a document about pineapple",
        "This is a document about oranges"
    ]
)"""

'collection.add(\n    ids=["id1", "id2"],\n    documents=[\n        "This is a document about pineapple",\n        "This is a document about oranges"\n    ]\n)'

In [102]:
"""from pprint import pprint
results = collection.query(
    query_texts=["This is a query document about hawai"], # Chroma will embed this for you
    n_results=2, # how many results to return
    where_document={'$contains':'pineapple'}
)"""


In [103]:
"""pprint(results)"""

{'data': None,
 'distances': [[]],
 'documents': [[]],
 'embeddings': None,
 'ids': [[]],
 'included': ['metadatas', 'documents', 'distances'],
 'metadatas': [[]],
 'uris': None}


In [127]:
import polars as pl

In [128]:
articels = pl.read_csv('Articles.csv',encoding='ISO-8859-1').with_row_index(offset=1)

In [129]:
articels.head()

index,Article,Date,Heading,NewsType
u32,str,str,str,str
1,"""KARACHI: The Sindh government …","""1/1/2015""","""sindh govt decides to cut publ…","""business"""
2,"""HONG KONG: Asian markets start…","""1/2/2015""","""asia stocks up in new year tra…","""business"""
3,"""HONG KONG: Hong Kong shares o…","""1/5/2015""","""hong kong stocks open 0.66 per…","""business"""
4,"""HONG KONG: Asian markets tumbl…","""1/6/2015""","""asian stocks sink euro near ni…","""business"""
5,"""NEW YORK: US oil prices Monday…","""1/6/2015""","""us oil prices slip below 50 a …","""business"""


In [130]:
!pip install openai

In [108]:
key='OPENAI_API_KEY'


In [131]:
import chromadb.utils.embedding_functions as embedding_functions

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-small-en-v1.5"
)

In [132]:
len(articels)

2692

In [133]:
n=50
a=articels[:n]

In [112]:
len(a)

50

In [134]:
articels_list=a['Article'][1:].to_list()
vec=embedding_function(articels_list)#we tird with i article now we do that with other 49

In [114]:
#we require ids as well sp id from 2 to 50
ids=[f"id{x}" for x in a['index'][1:].to_list()]

In [115]:
ids

['id2',
 'id3',
 'id4',
 'id5',
 'id6',
 'id7',
 'id8',
 'id9',
 'id10',
 'id11',
 'id12',
 'id13',
 'id14',
 'id15',
 'id16',
 'id17',
 'id18',
 'id19',
 'id20',
 'id21',
 'id22',
 'id23',
 'id24',
 'id25',
 'id26',
 'id27',
 'id28',
 'id29',
 'id30',
 'id31',
 'id32',
 'id33',
 'id34',
 'id35',
 'id36',
 'id37',
 'id38',
 'id39',
 'id40',
 'id41',
 'id42',
 'id43',
 'id44',
 'id45',
 'id46',
 'id47',
 'id48',
 'id49',
 'id50']

In [116]:
colection=chroma_client.get_or_create_collection(name="article",embedding_function=embedding_function)


In [117]:
print(len(vec))

49


In [118]:
print(type(vec[0]))

<class 'numpy.ndarray'>


In [119]:
print(vec[:3])

[array([-1.05618900e-02,  1.36765065e-02, -5.25603117e-03,  1.19577087e-02,
        6.27789870e-02,  2.09987490e-03, -4.18350622e-02,  2.98267938e-02,
        4.10998538e-02, -2.20098887e-02,  7.88067058e-02, -5.56075126e-02,
        5.13998745e-03,  5.27972728e-02, -7.18060210e-02,  3.38130482e-02,
       -6.39285147e-02, -1.40443861e-01, -5.43692075e-02, -2.04290729e-02,
       -9.20630321e-02, -3.81386206e-02, -8.86870641e-03, -1.11621760e-01,
        5.45770526e-02,  7.80826376e-04,  6.40710592e-02, -3.58671993e-02,
        7.38625461e-03, -1.34495020e-01, -5.86148053e-02, -6.91246474e-03,
        4.62244116e-02,  2.98128240e-02,  1.76614393e-02,  1.61502119e-02,
       -3.81089933e-03,  1.18261026e-02, -1.52446581e-02, -5.61921485e-02,
       -4.05285805e-02,  3.70053463e-02,  4.44518887e-02, -5.42665645e-03,
        7.67869353e-02, -1.73663348e-02,  7.65157640e-02,  4.54331227e-02,
       -2.39875969e-02,  9.89034958e-03,  5.59913516e-02,  1.70905627e-02,
        5.98219037e-02, 

In [120]:
collection.add(
    documents=articels_list,
    ids=ids,
    embeddings=vec

)

In [121]:
collection.count()

50

In [122]:
q='The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction '

query_embeddings=embedding_function([q])
collection.query(
    query_embeddings=query_embeddings,
    n_results=3,

)

{'ids': [['id1', 'id17', 'id24']],
 'embeddings': None,
 'documents': [['KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported.Sources said reduction in fares will be applicable on public transport, rickshaw, taxi and other means of traveling.Meanwhile, Karachi Transport Ittehad (KTI) has refused to abide by the government decision.KTI President Irshad Bukhari said the commuters are charged the lowest fares in Karachi as compare to other parts of the country, adding that 80pc vehicles run on Compressed Natural Gas (CNG). Bukhari said Karachi transporters will cut fares when decrease in CNG prices will be made.                        \n\n\n\n\n\n\n\n\n\n\n',
   'ISLAMABAD: The National Electric Power Regulatory Authority (NEPRA) on Wednesday said that a notification has been issued after November\x92s fuel adjustment regarding a relief on electricity c

In [135]:
a[23]['Article'][0]

'ISLAMABAD: In a move to give relief to consumers, sources in the Finance Ministry said on Tuesday that the price of petrol and petroleum products are expected to decrease further from February 1.According to sources, the price of petrol is expected to be slashed by Rs 10 per litre, High Speed Diesel by Rs 8.50 per litre, Light Diesel by Rs 11 per litre, HOBC by Rs 14 per litre, and Kerosene by Rs 12 per litre.Global crude oil prices have fallen by 50 percent since June 2014, and to provide consistent relief to consumers, the Pakistan government has decreased the price of petrol by Rs 29 since the last four months and brought the price of Diesel down by Rs 23 in the same time frame.Fuel crisis in the country began last week when Pakistan State Oil (PSO) was forced to slash imports because banks refused to extend any more credit to the government-owned company, which supplies 80 percent of the country´s oil.The shortfall led to long queues of angry motorists at petrol stations, though t

In [137]:
#to save the embeddings
chorma_client=chromadb.PersistentClient(path='./vectordb')

#CRUD

In [142]:
client=chromadb.Client()

In [146]:
collection=client.get_or_create_collection(name='anushka',metadata={"description":"..." })

#get collection


In [156]:
collections=client.list_collections()
print(collections)

[Collection(name=Article), Collection(name=Documents), Collection(name=anushkathakur), Collection(name=article)]


list_collection can have only 100 collection (eg here i have 4) so if i have more than 100 collection we cane do this:


In [152]:
batch_size=100
offset=0
all_collections=[]
while True:
  collect_btch=client.list_collections(limit=batch_size,offset=offset)
  if not collect_btch:
    break
  all_collections.extend(collect_btch)
  offset+=batch_size
print(all_collections)


[Collection(name=Article), Collection(name=Documents), Collection(name=anushka), Collection(name=article)]



get that collection

modify it


In [155]:
collection.modify(

     name="anushkathakur"

)

#delete


In [157]:
client.delete_collection(name="anushkathakur")

In [158]:
collection.peek(
    limit=2
)

NotFoundError: Collection [ab9046ce-8ae7-4979-803d-107061095f36] does not exist.